# 01.2 - 01.4 — Indexing, broadcasting, and the bugs that do not raise

**The question this block answers:** given two shapes, what comes out — and when does PyTorch silently give me a *plausible* tensor instead of an error?

**Prereqs:** 00.4 (matmul + shape algebra).

**Interview one-liners**
- **Basic slicing gives a view; advanced indexing gives a copy.** Writing into a view silently mutates the parent. This is the opposite of Python's `list[:]`.
- `x[:, -1, :]` **drops** the axis, `x[:, -1:, :]` **keeps** it. Picking the wrong one is the most common generation-loop bug.
- Broadcasting aligns shapes **from the right**; each pair must be **equal, or one of them 1**. Missing leading dims are treated as 1.
- The dangerous case is not the error — it is the shapes that broadcast successfully and mean something you did not intend. `(3,)` and `(3,1)` both "work" against a `(3,3)` and give **different answers**.
- `keepdim=True` exists precisely to stop a reduction from collapsing an axis you were about to broadcast against.

In [1]:
import sys
sys.path.insert(0, '..')

import torch
from common import show

torch.manual_seed(0)
print('torch', torch.__version__)

torch 2.10.0


---
# 01.2 — Indexing and slicing

## Experiment 1 — what each index expression does to the shape

One tensor, shape `(2, 3, 4)`. Read it as **2 sequences, 3 tokens each, 4 features per token** —
the `(B, T, d)` layout you already met in embeddings.

In [2]:
x = torch.arange(24).reshape(2, 3, 4)
print(x, '\n')

for expr, t in [
    ("x[0]",          x[0]),           # first sequence
    ("x[:, 1]",       x[:, 1]),        # token 1 of every sequence
    ("x[..., -1]",    x[..., -1]),     # last feature of every token
    ("x[:, :, 1:3]",  x[:, :, 1:3]),   # a slice of features
]:
    print(f'{expr:16s} -> {tuple(t.shape)}')

tensor([[[ 0,  1,  2,  3],
         [ 4,  5,  6,  7],
         [ 8,  9, 10, 11]],

        [[12, 13, 14, 15],
         [16, 17, 18, 19],
         [20, 21, 22, 23]]]) 

x[0]             -> (3, 4)
x[:, 1]          -> (2, 4)
x[..., -1]       -> (2, 3)
x[:, :, 1:3]     -> (2, 3, 2)


### Reading the output
An **integer** index removes that axis. A **slice** (`1:3`) keeps it, with a smaller size.
`...` means "all the axes I did not mention".

That single distinction — integer removes, slice keeps — is the whole of experiment 3.

## Experiment 2 — view or copy? Mutate and find out

The reliable test is not reading the docs, it is writing into the result and checking the parent.

In [3]:
# --- basic slicing ---------------------------------------------------
x = torch.arange(24).reshape(2, 3, 4)
v = x[0]                 # basic slicing
v[0, 0] = 999            # write into the RESULT
print('after writing into x[0]      : x[0,0,0] =', x[0, 0, 0].item())
print('shares storage with parent?  :', v.data_ptr() == x.data_ptr())

# --- boolean mask ----------------------------------------------------
x = torch.arange(24).reshape(2, 3, 4)
c = x[x > 10]            # advanced (boolean) indexing
c[0] = -1
print('\nafter writing into x[x>10]   : x[0,2,3] =', x[0, 2, 3].item(), '(unchanged)')
print('boolean result shape         :', tuple(c.shape), '<- always flattened')

# --- integer-array indexing -----------------------------------------
x = torch.arange(24).reshape(2, 3, 4)
f = x[[0, 1], [0, 2]]    # advanced (integer-array) indexing
f[0, 0] = -1
print('\nafter writing into x[[0,1],[0,2]]: x[0,0,0] =', x[0, 0, 0].item(), '(unchanged)')

after writing into x[0]      : x[0,0,0] = 999
shares storage with parent?  : True

after writing into x[x>10]   : x[0,2,3] = 11 (unchanged)
boolean result shape         : (13,) <- always flattened

after writing into x[[0,1],[0,2]]: x[0,0,0] = 0 (unchanged)


### Reading the output
- `x[0]` is a **view** — `999` appeared in the parent, and the storage pointer is identical.
- `x[x > 10]` and `x[[0,1],[0,2]]` are **copies** — the parent is untouched.

The rule: **basic slicing → view, advanced indexing (boolean or integer-array) → copy.**

Why you care: in a training loop, writing into a view of your activations mutates them
underneath autograd. The reverse trap is assuming a copy is a view and wondering why your
in-place update had no effect.

## Experiment 3 — the one that will bite you: rank drop vs rank kept

In a generation loop you want **the last token's** output. There are two ways to write it,
they differ by one character, and both run.

In [4]:
logits = torch.randn(2, 4, 50)      # (B=2 sequences, T=4 tokens, V=50 vocab)

drop = logits[:, -1, :]             # integer index  -> axis REMOVED
keep = logits[:, -1:, :]            # slice of size 1 -> axis KEPT

print('logits          ', tuple(logits.shape))
print('logits[:, -1, :]', tuple(drop.shape), ' <- rank 2')
print('logits[:, -1:, :]', tuple(keep.shape), '<- rank 3')
print('\nsame numbers?', torch.equal(drop, keep.squeeze(1)))

logits           (2, 4, 50)
logits[:, -1, :] (2, 50)  <- rank 2
logits[:, -1:, :] (2, 1, 50) <- rank 3

same numbers? True


### Reading the output
**Identical numbers, different rank.** Neither raises.

Which you want depends on what comes next. `torch.softmax(..., dim=-1)` is happy with either.
But feed the rank-3 version into something expecting `(B, V)` and you get a silent broadcast
instead of an error — which is experiment 6.

> **Predict-first:** you are writing the generation loop. You need the last step's logits to
> sample the next token. Which form do you want, and what would break if you picked the other?
>
> _(write your answer here before moving on)_

---
# 01.3 — Broadcasting rules

## The rule, in full

Line the two shapes up **from the right**. For each column:

1. equal → fine
2. one of them is `1` → that one is stretched
3. anything else → error

A shape that runs out of dimensions on the left is padded with `1`s.

```
      (2, 3, 4)          (3, 1)          (3, 4)
      (      4)   ->     (1, 4)   ->     (2, 4)
      ---------          ------          ------
      (2, 3, 4)  OK      (3, 4)  OK      ERROR  (3 vs 2, neither is 1)
```

Nothing is copied. PyTorch fakes the stretch with a stride of 0.

In [5]:
cases = [
    ((2, 3, 4), (4,)),        # trailing dim matches, leading padded
    ((3, 1),    (1, 4)),      # each stretches the other's 1
    ((2, 1, 4), (3, 1)),      # stretch on two different axes at once
    ((3, 4),    (2, 4)),      # 3 vs 2, neither is 1 -> error
    ((3, 4),    (4, 3)),      # right-aligned: 4vs3 -> error
]

for sa, sb in cases:
    try:
        out = torch.zeros(*sa) + torch.zeros(*sb)
        print(f'{str(sa):12s} + {str(sb):8s} -> {tuple(out.shape)}')
    except RuntimeError as e:
        print(f'{str(sa):12s} + {str(sb):8s} -> ERROR')

(2, 3, 4)    + (4,)     -> (2, 3, 4)
(3, 1)       + (1, 4)   -> (3, 4)
(2, 1, 4)    + (3, 1)   -> (2, 3, 4)
(3, 4)       + (2, 4)   -> ERROR
(3, 4)       + (4, 3)   -> ERROR


### Reading the output
Case 3 is worth staring at: `(2,1,4)` and `(3,1)` become `(2,3,4)`. The second shape is padded
to `(1,3,1)`, then **both** tensors get stretched — the first along its middle axis, the second
along its first and last. Neither had `(2,3,4)` worth of data.

The last case is the one people misread: `(3,4)` and `(4,3)` look symmetric, but alignment is
from the **right**, so it compares `4` against `3` and fails.

## Experiment 5 — broadcasting does not copy

A stretched axis gets **stride 0**: "to move one step along this axis, move 0 bytes."
The same memory is read repeatedly.

In [6]:
col = torch.arange(3.).reshape(3, 1)        # (3,1)
wide = col.expand(3, 4)                     # broadcast to (3,4), NO copy

print('col   shape', tuple(col.shape),  ' stride', col.stride())
print('wide  shape', tuple(wide.shape), ' stride', wide.stride(), '  <- the 0')
print('same storage?', col.data_ptr() == wide.data_ptr())
print()
print(wide)
print('\nelements in storage:', col.numel(), ' elements the view presents:', wide.numel())

col   shape (3, 1)  stride (1, 1)
wide  shape (3, 4)  stride (1, 0)   <- the 0
same storage? True

tensor([[0., 0., 0., 0.],
        [1., 1., 1., 1.],
        [2., 2., 2., 2.]])

elements in storage: 3  elements the view presents: 12


### Reading the output
12 elements presented, 3 actually stored, stride `0` on the stretched axis, same pointer.

This is why broadcasting is cheap, and also why a broadcast tensor is **not contiguous** —
which is what makes `.view()` fail on it later (01.5).

---
# 01.4 — The bug drill: shapes that broadcast and are still wrong

Everything above raised an error when it was wrong. **These do not.** They produce a tensor of
a believable shape, full of wrong numbers, and your loss still goes down — just to a worse place.

## Experiment 6 — the `keepdim` bug (this is the softmax bug)

You want to subtract each **row's** maximum from that row. Two ways to get the max, one character apart.

In [7]:
scores = torch.tensor([[1., 2., 3.],
                       [10., 20., 30.],
                       [100., 200., 300.]])

mx_keep = scores.max(dim=-1, keepdim=True).values    # (3,1)
mx_drop = scores.max(dim=-1).values                  # (3,)

print('mx_keep', tuple(mx_keep.shape), mx_keep.flatten().tolist())
print('mx_drop', tuple(mx_drop.shape), mx_drop.tolist())
print()
print('scores - mx_keep  ->', tuple((scores - mx_keep).shape))
print(scores - mx_keep)
print()
print('scores - mx_drop  ->', tuple((scores - mx_drop).shape), '  same shape, NO error')
print(scores - mx_drop)

mx_keep (3, 1) [3.0, 30.0, 300.0]
mx_drop (3,) [3.0, 30.0, 300.0]

scores - mx_keep  -> (3, 3)
tensor([[  -2.,   -1.,    0.],
        [ -20.,  -10.,    0.],
        [-200., -100.,    0.]])

scores - mx_drop  -> (3, 3)   same shape, NO error
tensor([[  -2.,  -28., -297.],
        [   7.,  -10., -270.],
        [  97.,  170.,    0.]])


### Reading the output
Both give `(3, 3)`. **Neither raises.** The numbers are completely different.

- `(3,1)` aligns as a **column** — each row gets its own max subtracted. Correct.
- `(3,)` is padded to `(1,3)` and aligns as a **row** — every row gets the *same three* maxima
  subtracted, one per column. Nonsense.

Look at the correct version: every row's largest entry becomes `0`, which is exactly what the
stable-softmax trick needs. The broken version has positive values left in it.

**This is the single most common shape bug in a hand-written attention implementation**, and you
will meet it again in 03.3 where the softmax max-subtraction is over a `(B,H,T,T)` tensor.

## Experiment 7 — mask orientation: `(T,1)` vs `(1,T)`

Same numbers, transposed shape. Against a `(T,T)` score matrix both broadcast cleanly,
and they mask completely different things.

In [8]:
T = 4
scores = torch.zeros(T, T)
flags  = torch.tensor([0., 0., 1., 1.])      # "positions 2 and 3 are padding"

as_row = scores.masked_fill(flags.reshape(1, T) == 1, float('-inf'))
as_col = scores.masked_fill(flags.reshape(T, 1) == 1, float('-inf'))

print('as_row  (1,T) — masks COLUMNS:')
print(as_row)
print('\nas_col  (T,1) — masks ROWS:')
print(as_col)
print('\nboth shapes:', tuple(as_row.shape), tuple(as_col.shape), ' - no error either way')

as_row  (1,T) — masks COLUMNS:
tensor([[0., 0., -inf, -inf],
        [0., 0., -inf, -inf],
        [0., 0., -inf, -inf],
        [0., 0., -inf, -inf]])

as_col  (T,1) — masks ROWS:
tensor([[0., 0., 0., 0.],
        [0., 0., 0., 0.],
        [-inf, -inf, -inf, -inf],
        [-inf, -inf, -inf, -inf]])

both shapes: (4, 4) (4, 4)  - no error either way


### Reading the output
`(1,T)` blanks out **columns** — "nobody may attend *to* these keys". That is a padding mask.

`(T,1)` blanks out **rows** — "these queries may attend to nothing at all". Every one of those
rows is now entirely `-inf`, and after softmax it becomes `nan`.

Same data, one `reshape` apart, and one of them poisons your loss with `nan` while the other is
correct. You will meet this exact pair in 03.4.

## The habit that prevents all of this

`common/shapes.py` exists for this. Print the shape of everything, every time, while you are
learning — and `assert_shape` immediately after any reduction, reshape or transpose.

In [9]:
from common import assert_shape

scores = torch.randn(3, 5)

mx = scores.max(dim=-1, keepdim=True).values
assert_shape(mx, (3, 1), 'row max')        # passes
print('row max shape OK')

bad = scores.max(dim=-1).values
try:
    assert_shape(bad, (3, 1), 'row max')   # catches the bug AT the line that made it
except AssertionError as e:
    print('caught:', e)

row max shape OK
caught: row max: expected shape (3, 1), got (3,)


---
## Challenge — predict every line before running

Write your answers first. Four of these six succeed; two raise.

| # | Expression | Shape or ERROR? |
| --- | --- | --- |
| 1 | `(8, 1, 6) + (7, 1)` | _(write)_ |
| 2 | `(5, 4) + (4,)` | _(write)_ |
| 3 | `(5, 4) + (5,)` | _(write)_ |
| 4 | `(2, 3, 4) * (2, 1, 1)` | _(write)_ |
| 5 | `(4, 3) + (4, 1, 3)` | _(write)_ |
| 6 | `(2, 3) + (3, 2)` | _(write)_ |

And one in words: for #3, if it raises — what single change to the *second* shape would make it
do what a person writing `(5,4) + (5,)` probably intended?

In [ ]:
# run AFTER writing your predictions above
for sa, sb in [((8,1,6),(7,1)), ((5,4),(4,)), ((5,4),(5,)),
               ((2,3,4),(2,1,1)), ((4,3),(4,1,3)), ((2,3),(3,2))]:
    try:
        print(f'{str(sa):12s} + {str(sb):10s} -> {tuple((torch.zeros(*sa) + torch.zeros(*sb)).shape)}')
    except RuntimeError:
        print(f'{str(sa):12s} + {str(sb):10s} -> ERROR')